# 实验二 · Host 与 Device 的内存和数据传输 —— 锁页内存与传输带宽模型

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐⭐ 进阶　|　**预计时长**：50–60 分钟

一个只搬几十字节的程序无需关心传输耗时。真实应用不是这样：一次 ResNet-50 推理要搬进约 150 KB 的图片（`uint8`，224×224×3）、搬出 4 KB 的分类结果，一次大模型推理要搬进上百 MB 的 KV Cache。本实验回答两个问题：数据从主机搬到设备要花多少时间，这个时间由什么决定。

回答的方式是测出一条曲线。以传输块大小为横轴、有效带宽为纵轴扫描一遍，就能从中解出两个常数：单次传输的固定开销 $\alpha$ 与渐近带宽 $\beta$。这两个常数刻画的是本机这条传输路径本身：$\alpha$ 决定一次传输至少要花多少时间，$\beta$ 决定传输量足够大时能跑多快，由二者导出的半带宽规模 $S_{1/2} = \alpha\beta$ 决定传输块至少该切多大。

> **实验说明**
> 1. 本实验的核心内容有四点：两侧内存的申请与释放、可分页内存与锁页内存的差别、同步复制与异步复制的差别，以及 $T = \alpha + S/\beta$ 这一传输时间模型。
> 2. 本实验采用**四个版本的两因素设计**：主机内存类型（可分页／锁页）与复制接口（同步／异步）两两组合，用以分离两个因素各自的作用。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**。
> 5. 四个版本的测量代码写在同一个 `.cpp` 文件中，由一条 `g++` 命令编译为单个可执行程序，一次运行产出全部对照数据。
> 6. 本实验的耗时一律用**主机侧墙钟时间**测量，不使用设备侧计时。
> 7. 本实验默认读者已经掌握 acl 应用的七步框架与资源配对规则。


## 🎯 学习目标

完成本实验后，学生应能够：

- 区分可分页内存与锁页内存，说明后者为何能让设备直接通过 DMA 通道访问
- 掌握 `aclrtMalloc` / `aclrtFree` 与 `aclrtMallocHost` / `aclrtFreeHost` 两对接口的适用范围，并说明二者不可互换的原因
- 说出 `aclrtMemcpy` 五个参数的含义，指出两个 size 参数的区别
- 说明 `aclrtMemcpyAsync` 真正异步执行的前提条件，以及不满足该条件时接口的实际行为
- 说明 `aclrtMemcpyAsync` 对源地址与目的地址的 64 字节对齐要求，以及哪些接口会自动满足该要求
- 正确设计一次带宽测量：首次触碰、预热、多次重复取平均、同步之后再停止计时
- 从实测曲线的两端读出 $T = \alpha + S/\beta$ 中的固定开销 $\alpha$ 与渐近带宽 $\beta$，并说明各自读自哪一段
- 由 $\alpha$ 与 $\beta$ 算出半带宽规模 $S_{1/2}$，并说明它为什么是判断传输块该切多大的判据
- 说明本实验的测量方法与第二章存储层次实验的同源关系


## 🗺️ 学习路径

1. **准备阶段**：理解两侧内存相互独立所带来的显式搬运需求，以及传输开销为什么必须单独测量
2. **概念建立**：两侧内存的申请与释放，可分页内存与锁页内存的差别，以及对齐要求
3. **接口辨析**：同步复制与异步复制的区别，以及异步接口真正异步所需要的前提条件
4. **测量方法**：建立一次带宽测量的完整做法，理解首次触碰、预热与重复取平均各自排除的是什么
5. **对照设计**：由主机内存类型与复制接口两个因素组合出四个版本，使每一项差异都可以归因
6. **模型建立**：把传输耗时写成固定开销与传输量之和，从曲线两端解出两个常数，并由它们算出半带宽规模
7. **结果分析**：区分随平台变化与不随平台变化的结论，把工程判据写成带条件的形式


## 1. 背景与动机：为什么传输开销必须单独测量

第六章的实验测量的都是同一件事：核函数在设备上的执行时间。但一个真实应用的耗时不止于此——主机与设备各自拥有独立的内存空间，数据不会自己出现在设备上。

<img src="images/07.02_host_device.png" alt="主机与设备各自拥有独立的内存" height="430">

图中主机与设备各有自己的内存与互联控制器，两者之间只通过总线相连。**任何一次计算，数据都必须先跨过这条总线，算完再跨回来。** 这段路程的代价，就是本实验要测量的对象。图中 Device 侧还画出了任务调度器与几类执行单元，本实验不涉及它们，只关心两侧的内存与其间的总线。

第六章已经给出过一个提示：同一个向量加法，纯核函数耗时与端到端耗时相差一到两个数量级，差额正是两次跨总线搬运。当时的结论是计算速度快不等于整体收益大，但没有回答搬运本身要花多少时间。本实验把这个问题单独拿出来测。

### 1.1 与第二章的同源关系

测量方法本身并不新。第二章测量内存与 Cache 之间的带宽时，用的是同一套方法：

<!-- 与第二章的对照 -->
<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 维度 | 第二章 存储层次 | 第七章 实验二 |
| --- | --- | --- |
| 被测介质 | DRAM 与各级 Cache | 主机内存与设备内存之间的总线（PCIe 或 HCCS） |
| 自变量 | 访问块大小 | 传输块大小 |
| 因变量 | 有效带宽 | 有效带宽 |
| 模型 | $T = \alpha + S/\beta$ | $T = \alpha + S/\beta$ |
| $\alpha$ 的量级 | 纳秒 | 微秒 |
| $\beta$ 的量级 | 百 GB/s | 几十 GB/s |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">维度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第二章 存储层次</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">第七章 实验二</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">被测介质</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">DRAM 与各级 Cache</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机内存与设备内存之间的总线（PCIe 或 HCCS）</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">自变量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">访问块大小</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">传输块大小</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">因变量</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有效带宽</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">有效带宽</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$T = \alpha + S/\beta$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$T = \alpha + S/\beta$</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\alpha$ 的量级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">纳秒</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">微秒</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\beta$ 的量级</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">百 GB/s</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">几十 GB/s</td>
</tr>
</tbody>
</table>

同一套**微基准（microbenchmark）**方法，跨了约两个数量级的尺度。学生在本实验中要做的，是把第二章已经掌握的方法搬到一个新的硬件上，并解释量级差异的来源。

### 1.2 本实验不测什么

有两件事本实验刻意不做，划清测量的边界：

1. **不测设备侧耗时。** 本实验全部用主机侧墙钟时间，因此测到的是应用视角的端到端传输耗时，其中包含接口调用与同步等待。
2. **不做传输与计算的重叠。** 本实验的每一次测量都是下发一次、等它完成，测的是单次传输的代价。异步接口在本实验中只用来说明它的前提条件。


## 2. 两侧内存的申请与释放

### 2.1 Device 内存

Device 内存由 `aclrtMalloc` 申请、由 `aclrtFree` 释放。有四条规则需要记住：

1. **申请到的内存首地址 64 字节对齐**，size 会被向上对齐成 32 字节整数倍后再多加 32 字节。
2. **size 不能为 0**，否则接口返回 `ACL_ERROR_INVALID_PARAM`。
3. **申请到的内存不会被初始化**，Runtime API 参考建议使用前先调用 `aclrtMemset` 清除其中的随机值。本实验在 `aclrtMalloc` 之后立即调用一次 `aclrtMemset`（见 §7.5）。
4. **频繁申请与释放会损耗性能**，Runtime API 参考建议提前预分配或自行做二次管理。本实验的做法是按最大规模一次性申请，所有规模的测量共用同一块内存。

第三个参数是内存分配规则 `aclrtMemMallocPolicy`，本实验用 `ACL_MEM_MALLOC_HUGE_FIRST`：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 枚举值 | 含义 |
| --- | --- |
| `ACL_MEM_MALLOC_HUGE_FIRST` | 大页优先，申请粒度 2 MB。申请量不超过 1 MB 时仍按普通页申请；超过 1 MB 时优先大页，大页不足则回退普通页 |
| `ACL_MEM_MALLOC_HUGE_ONLY` | 仅申请大页，大页不足则返回错误 |
| `ACL_MEM_MALLOC_NORMAL_ONLY` | 仅申请普通页，普通页不足则返回错误 |
| `ACL_MEM_MALLOC_HUGE1G_ONLY` | 仅申请 1 GB 粒度的大页。相比 2 MB 粒度，同样 1 GB 内存的页表数从 512 项降到 1 项，可以扩大 TLB 覆盖的地址范围，提升离散访问性能 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">枚举值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEM_MALLOC_HUGE_FIRST</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">大页优先，申请粒度 2 MB。申请量不超过 1 MB 时仍按普通页申请；超过 1 MB 时优先大页，大页不足则回退普通页</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEM_MALLOC_HUGE_ONLY</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">仅申请大页，大页不足则返回错误</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEM_MALLOC_NORMAL_ONLY</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">仅申请普通页，普通页不足则返回错误</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEM_MALLOC_HUGE1G_ONLY</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">仅申请 1 GB 粒度的大页。相比 2 MB 粒度，同样 1 GB 内存的页表数从 512 项降到 1 项，可以扩大 TLB 覆盖的地址范围，提升离散访问性能</td>
</tr>
</tbody>
</table>

四个枚举值的含义出自 Runtime API 参考。本实验采用 `ACL_MEM_MALLOC_HUGE_FIRST`：本实验一次申请的最大规模为 128 MB，属于大块内存，优先用大页可以减少页表项数量；大页不足时它会自动退回普通页，因此不会因为大页耗尽而申请失败。

**与第二章的呼应**：大页的作用是减少页表项数量、提高 TLB 命中率，这与第二章讨论的 TLB 与页表机制完全同源，只是这里的对象换成了设备内存。

### 2.2 Host 内存：可分页与锁页

主机侧有两类内存，差别是本实验的核心。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 维度 | 可分页内存（Pageable） | 锁页内存（Page-Locked / Pinned） |
| --- | --- | --- |
| 申请与释放 | `malloc`、`mmap` / `free`、`munmap` | `aclrtMallocHost` / `aclrtFreeHost` |
| 管理者 | 操作系统统一管理 | Runtime 管理 |
| 内存压力下的行为 | **会被换出到交换空间** | 虚拟页与物理页的映射关系固定，生命周期内**不会被换出** |
| 传输路径 | 数据**先复制到中转缓冲区**，再经 DMA 通道传到 Device | **直接经 DMA 通道传输，无需经过中转缓冲区** |
| 首地址对齐 | 由分配器决定，无保证 | 由系统保证 **64 字节对齐** |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">维度</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">可分页内存（Pageable）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">锁页内存（Page-Locked / Pinned）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">申请与释放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>malloc</code>、<code>mmap</code> / <code>free</code>、<code>munmap</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMallocHost</code> / <code>aclrtFreeHost</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">管理者</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">操作系统统一管理</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Runtime 管理</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内存压力下的行为</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>会被换出到交换空间</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">虚拟页与物理页的映射关系固定，生命周期内<strong>不会被换出</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">传输路径</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">数据<strong>先复制到中转缓冲区</strong>，再经 DMA 通道传到 Device</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>直接经 DMA 通道传输，无需经过中转缓冲区</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首地址对齐</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由分配器决定，无保证</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由系统保证 <strong>64 字节对齐</strong></td>
</tr>
</tbody>
</table>

使用锁页内存的三条好处：

1. 设备可以直接通过 DMA 访问主机内存，无需经过缓冲区，可以提供更好的传输性能；
2. 数据搬运过程**无需 CPU 参与**，可以实现数据的异步传输；
3. 数据的异步传输，使**传输过程和计算过程可以相互掩盖**，减少传输与计算的整体时长。

第一条是本实验要验证的；第二、三条要在传输与计算重叠的场景中才体现出来，本实验不涉及。

> ⚠️ 锁页内存不是越多越好。**使用 `aclrtMallocHost` 分配过多的锁页内存，将导致操作系统用于分页的物理内存变少，从而降低系统整体的性能。** 本实验按最大传输规模申请，实测完即释放。

### 2.3 一条容易忽略的对齐要求

`aclrtMemcpyAsync` 的约束是：**调用本接口进行内存复制时，源地址和目的地址都必须 64 字节对齐。** 这条约束是本实验的可分页缓冲区不用 `malloc` 的唯一理由，

这条约束对四个版本的影响并不相同：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 内存来源 | 首地址对齐 | 能否直接用于 `aclrtMemcpyAsync` |
| --- | --- | --- |
| `aclrtMalloc` | 64 字节（接口保证） | 可以 |
| `aclrtMallocHost` | 64 字节（接口保证） | 可以 |
| `malloc` | 通常 16 字节，**无保证** | **不可以**，需显式对齐 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内存来源</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">首地址对齐</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">能否直接用于 <code>aclrtMemcpyAsync</code></th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMalloc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">64 字节（接口保证）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMallocHost</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">64 字节（接口保证）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>malloc</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">通常 16 字节，<strong>无保证</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不可以</strong>，需显式对齐</td>
</tr>
</tbody>
</table>

因此本实验的可分页缓冲区不用 `malloc` 而用 `std::aligned_alloc(64, bytes)` 申请。


## 3. 内存复制接口

### 3.1 `aclrtMemcpy` 的五个参数

```cpp
aclError aclrtMemcpy(void *dst, size_t destMax, const void *src, size_t count,
                     aclrtMemcpyKind kind);
```

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 参数 | 含义 | 易错点 |
| --- | --- | --- |
| `dst` | 目的内存地址 | — |
| `destMax` | **目的内存的最大长度**，单位字节 | 与 `count` 语义不同，不是同一个数的重复 |
| `src` | 源内存地址 | — |
| `count` | **实际复制的长度**，单位字节 | 必须不大于 `destMax` |
| `kind` | 复制类型 | **当前版本是预留参数**，见下 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">参数</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">易错点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>dst</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">目的内存地址</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>destMax</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>目的内存的最大长度</strong>，单位字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">与 <code>count</code> 语义不同，不是同一个数的重复</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>src</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">源内存地址</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">—</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>count</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>实际复制的长度</strong>，单位字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">必须不大于 <code>destMax</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>kind</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">复制类型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>当前版本是预留参数</strong>，见下</td>
</tr>
</tbody>
</table>

两个 size 参数的存在是一种边界保护：接口内部会用 `destMax` 检查是否越界写入。二者写反在小规模测试中往往不会立刻出错，一旦复制长度超过目的缓冲区就是内存越界。

关于 `kind`，Runtime API 参考的说明是：**这是预留参数，配置枚举值中的值无效，系统内部会根据源内存地址指针、目的内存地址指针判断是否可以将源地址的数据复制到目的地址，如果不可以，则系统会返回报错。**

`aclrtMemcpyKind` 的完整取值（本实验只用到其中两个）：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 枚举值 | 含义 |
| --- | --- |
| `ACL_MEMCPY_HOST_TO_HOST` | Host 内的内存复制 |
| `ACL_MEMCPY_HOST_TO_DEVICE` | Host 到 Device 的内存复制 |
| `ACL_MEMCPY_DEVICE_TO_HOST` | Device 到 Host 的内存复制 |
| `ACL_MEMCPY_DEVICE_TO_DEVICE` | Device 内或两个 Device 间的内存复制 |
| `ACL_MEMCPY_DEFAULT` | 由系统根据源、目的内存地址自行判断复制方向 |
| `ACL_MEMCPY_HOST_TO_BUF_TO_DEVICE` | Host 到 Device，但 Host 内存会暂存在 Runtime 管理的缓存中，接口调用成功后即可释放 Host 内存 |
| `ACL_MEMCPY_INNER_DEVICE_TO_DEVICE` | Device 内的内存复制 |
| `ACL_MEMCPY_INTER_DEVICE_TO_DEVICE` | 两个 Device 之间的内存复制 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">枚举值</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_HOST_TO_HOST</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 内的内存复制</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_HOST_TO_DEVICE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 到 Device 的内存复制</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_DEVICE_TO_HOST</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 到 Host 的内存复制</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_DEVICE_TO_DEVICE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 内或两个 Device 间的内存复制</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_DEFAULT</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由系统根据源、目的内存地址自行判断复制方向</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_HOST_TO_BUF_TO_DEVICE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Host 到 Device，但 Host 内存会暂存在 Runtime 管理的缓存中，接口调用成功后即可释放 Host 内存</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_INNER_DEVICE_TO_DEVICE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 内的内存复制</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>ACL_MEMCPY_INTER_DEVICE_TO_DEVICE</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两个 Device 之间的内存复制</td>
</tr>
</tbody>
</table>


### 3.2 同步与异步：一个有前提条件的异步

```cpp
aclError aclrtMemcpyAsync(void *dst, size_t destMax, const void *src,
                          size_t count, aclrtMemcpyKind kind,
                          aclrtStream stream);
```

比同步版本多了一个 Stream 参数。但**接口名中的 Async 并不意味着它总是异步的**。Runtime API 参考对它的说明是：

> 本接口中的 Host 内存支持锁页内存（例如通过 `aclrtMallocHost` 接口申请的内存）、非锁页内存（通过 `malloc` 接口申请的内存）。**当 Host 内存是锁页内存时，本接口是异步接口**，调用接口成功仅表示任务下发成功，不表示任务执行成功，调用本接口后，需调用同步等待接口确保内存复制的任务已执行完成；**当 Host 内存是非锁页内存时，本接口在内存复制任务完成后才返回。**

这段话的含义是：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| Host 内存类型 | `aclrtMemcpyAsync` 的实际行为 | 是否需要 `aclrtSynchronizeStream` |
| --- | --- | --- |
| 锁页内存 | **真正异步**，返回时任务只是被下发 | 必须调用，否则读到的是未完成的数据 |
| 可分页内存 | **退化为同步**，返回时复制已完成 | 调用它不会出错，但不起实质作用 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Host 内存类型</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>aclrtMemcpyAsync</code> 的实际行为</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">是否需要 <code>aclrtSynchronizeStream</code></th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">锁页内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>真正异步</strong>，返回时任务只是被下发</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">必须调用，否则读到的是未完成的数据</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可分页内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>退化为同步</strong>，返回时复制已完成</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调用它不会出错，但不起实质作用</td>
</tr>
</tbody>
</table>

原因在于 DMA 引擎的工作方式：DMA 需要在整个传输期间物理地址保持稳定，而可分页内存随时可能被操作系统换出。Runtime 面对可分页内存只能退回到先复制到内部的锁页中转缓冲区、再发起 DMA 这条路，而这次中转复制由 CPU 完成，必须同步等待。

**这条规则决定了本实验的版本设计**：异步接口与锁页内存不是两个独立的因素，前者以后者为前提。四个版本正是为了把这层关系测出来。

还有两条约束：

- Atlas A3 与 A2 系列产品**不支持异步的 Host 内复制**，`kind` 传 `ACL_MEMCPY_HOST_TO_HOST` 时接口返回 `ACL_ERROR_RT_FEATURE_NOT_SUPPORT`。本实验只做 H2D 与 D2H，不涉及 Host 内复制。
- 源地址与目的地址必须 64 字节对齐，见 §2.3。

`aclrtGetRunMode` 用于区分软件栈跑在 Host 还是 Device 的 Control CPU 上。按 Runtime API 参考，A2 与 A3 系列不支持 `ACL_DEVICE`，因此它在本实验中恒返回 `ACL_HOST`；程序把它打印出来，是因为它直接决定主机侧数据要不要显式复制：返回 `ACL_HOST` 时主机与设备内存独立，输入数据必须经 `aclrtMemcpy` 传到设备；返回 `ACL_DEVICE` 时可以直接申请并使用设备内存。常见的 `if (runMode == ACL_HOST)` 分支正是为这两种情形而写。

下图是基于 Runtime 编程的典型执行流程，它把一次异步任务的完整回路逐步编号画了出来：

<img src="images/07.02_task_scheduling.png" alt="基于 Runtime 编程的典型执行流程" width="560px">

主机侧下发任务（① 任务加入队列）之后接口即返回；设备侧的任务调度器取队列任务进行调度（②），按任务类型选择空闲的 CPU 核或 AI Core（③／③′），执行器完成后通知调度器（④／④′），调度器再反馈执行结果（⑤）；主机侧要取得这个结果，必须显式调用同步接口（⑥）。异步接口省下的，是主机线程在 ① 与 ⑥ 之间的等待时间；而这条回路本身的环节数，正是 §12 ④ 中异步路径固定开销高于同步路径的来源。本实验在 ① 之后紧接着就做 ⑥，因此只会测到这条回路的代价，测不到它的收益。

### 3.3 二维复制

除按线性区间复制外，Runtime 还提供了面向矩阵子块的二维复制接口：

```cpp
aclError aclrtMemcpy2d(void *dst, size_t dpitch, const void *src, size_t spitch,
                       size_t width, size_t height, aclrtMemcpyKind kind);
```

`spitch` 与 `dpitch` 分别是源、目的内存中相邻两行的地址距离，`width` 与 `height` 是待复制子块的宽和高。

<img src="images/07.02_memcpy_2d.png" alt="二维复制中 spitch 与 dpitch 的含义" width="693px">

图中源矩阵每行 10 个单位、目的矩阵每行 8 个单位，实际搬运的是 5 × 3 的子块，图下方注明了复制的顺序。动手练习第 2 题会把这张图的四个参数填成具体数字。

该接口**当前仅支持 `ACL_MEMCPY_HOST_TO_DEVICE` 与 `ACL_MEMCPY_DEVICE_TO_HOST` 两种类型**。它的价值在于把逐行调用 `aclrtMemcpy` 合并成一次调用，从而只付一次固定开销 $\alpha$。


## 4. 环境准备与检查

先把 CANN 的环境变量导入 Jupyter 进程，并创建代码目录。


In [ ]:
!mkdir -p src_memory

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


本实验的自检除了确认编译器与设备之外，还要打印**操作系统内核版本**：部分主机内存相关接口对内核版本有要求，排查异常时这是第一项要核对的信息。

本实验的程序只包含 `acl/acl.h`，因此按 §3.2 的规则**只需链接 Runtime 库**，不需要算子库。库名仍然从本机探测，不写死。


In [ ]:
import os, platform, shutil, subprocess

print("=" * 62)
print(" 一、平台与内核")
print("=" * 62)
print("操作系统   :", platform.system())
print("处理器架构 :", platform.machine())
print("内核版本   :", platform.release().split("-")[0],
      "（完整串 %s）" % platform.release())

print()
print("=" * 62)
print(" 二、CANN 与编译器")
print("=" * 62)
ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
print("ASCEND_HOME_PATH :", ascend_home or "⚠️  未设置")
print("g++              :", shutil.which("g++") or "⚠️  未找到")
print("npu-smi          :", shutil.which("npu-smi") or "⚠️  未找到")

lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]
rt_name = None
for name in ("acl_rt", "ascendcl"):
    for lib_dir in lib_dirs:
        if rt_name is None and os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
            rt_name = name

ACL_LIBDIRS = ["-L" + path for path in lib_dirs]
ACL_RT_LIB = ["-l" + rt_name] if rt_name else []
print()
print("库目录     :", " ".join(ACL_LIBDIRS))
print("Runtime 库 :", " ".join(ACL_RT_LIB) or "⚠️  未找到")

print()
if ascend_home and shutil.which("g++") and rt_name:
    print("✅ 环境就绪，可以开始实验。")
else:
    print("⚠️  环境不完整，请重新运行上一个单元格。")


## 5. 本实验的性能测量方法

### 5.1 两个指标与一个模型

本实验只有一个原始指标：把 $S$ 字节从一侧搬到另一侧所用的时间 $T$。由它导出有效带宽：

$$\text{BW} = \frac{S}{T}$$

单独看某一个 $S$ 下的带宽没有意义，因为它随 $S$ 变化。有意义的是整条曲线，以及从曲线中解出的两个常数。传输时间可以用一个只含两个参数的线性模型描述：

$$T(S) = \alpha + \frac{S}{\beta}$$

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 符号 | 含义 | 在曲线上的位置 |
| --- | --- | --- |
| $\alpha$ | **单次传输的固定开销**，与传输量无关。包含接口调用、描述符构造、DMA 任务下发与完成通知等 | $S \to 0$ 时的时间截距 |
| $\beta$ | **渐近带宽**，传输量足够大时能达到的稳定带宽 | $S \to \infty$ 时带宽曲线的水平渐近线 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">符号</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">含义</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">在曲线上的位置</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\alpha$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>单次传输的固定开销</strong>，与传输量无关。包含接口调用、描述符构造、DMA 任务下发与完成通知等</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S \to 0$ 时的时间截距</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$\beta$</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>渐近带宽</strong>，传输量足够大时能达到的稳定带宽</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S \to \infty$ 时带宽曲线的水平渐近线</td>
</tr>
</tbody>
</table>

由这两个常数可以导出一个实用的判据——**半带宽规模** $S_{1/2}$，即有效带宽达到渐近带宽一半时的传输量：

$$\frac{S_{1/2}}{\alpha + S_{1/2}/\beta} = \frac{\beta}{2}
\quad\Longrightarrow\quad S_{1/2} = \alpha\beta$$

这个模型有适用范围。它假定传输时间随传输量线性增长，而这一点并非总是成立——§12 ② 会讨论一个在部分平台上出现的反例：可分页内存跨过某个规模之后带宽不升反降，此时模型在整个量程上不再成立，只能分段使用。

$S_{1/2}$ 的工程含义是：**传输块小于这个规模时，时间主要花在固定开销上，搬运效率不到一半。** 这条判据在两个场合直接可用：其一，把一次大传输拆成多块下发时，每块必须明显大于 $S_{1/2}$，否则拆块多付的固定开销会盖过拆块本身的好处；其二，多次小传输应当先在主机侧拼成一次大传输，拼接的收益就是省下的那几份 $\alpha$。这个量在第二章讨论访问粒度时也出现过，两处的推导完全相同。

### 5.2 测量要点

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 要点 | 本实验的做法 | 不这样做的后果 |
| --- | --- | --- |
| **首次触碰** | 分配主机缓冲区后立即 `memset` 全部字节 | 首次传输会同时付出缺页中断的代价，小规模测量偏高 |
| **预热** | 每个测量点先跑 3 次不计时 | 首次调用包含一次性的初始化开销 |
| **多次重复取平均** | 按规模分三档：不超过 1 MB 重复 100 次，不超过 16 MB 重复 20 次，更大规模重复 5 次 | 单次测量的抖动可达数十个百分点 |
| **同步之后再停止计时** | 异步路径上先 `aclrtSynchronizeStream` 再停计时 | 测到的是任务下发耗时（几微秒），不是传输耗时 |
| **同一块内存反复使用** | 按最大规模申请一次，所有规模共用 | 把内存申请的开销混进传输耗时 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本实验的做法</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">不这样做的后果</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>首次触碰</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">分配主机缓冲区后立即 <code>memset</code> 全部字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首次传输会同时付出缺页中断的代价，小规模测量偏高</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>预热</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">每个测量点先跑 3 次不计时</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首次调用包含一次性的初始化开销</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多次重复取平均</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按规模分三档：不超过 1 MB 重复 100 次，不超过 16 MB 重复 20 次，更大规模重复 5 次</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">单次测量的抖动可达数十个百分点</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>同步之后再停止计时</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步路径上先 <code>aclrtSynchronizeStream</code> 再停计时</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">测到的是任务下发耗时（几微秒），不是传输耗时</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>同一块内存反复使用</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按最大规模申请一次，所有规模共用</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">把内存申请的开销混进传输耗时</td>
</tr>
</tbody>
</table>

第四条在本实验中尤其容易被忽略：由 §3.2 可知，`aclrtMemcpyAsync` 在锁页内存上是真正异步的，若不同步就停计时，v4 算出的数值会高出一个数量级——那不是传输变快，而是根本没有测到传输。

### 5.3 单位约定

本章的容量单位一律取二进制：1 KB = $2^{10}$ 字节，1 MB = $2^{20}$ 字节，1 GB = $2^{30}$ 字节；带宽单位 GB/s 相应地取 $2^{30}$ 字节每秒。


## 6. 版本设计总览

本实验采用**两因素两水平的全组合设计**：主机内存类型（可分页／锁页）× 复制接口（同步／异步），共四个版本。

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | Host 内存 | 复制接口 | 接口的实际行为 | 引入的新概念 |
| --- | --- | --- | --- | --- |
| v1 | `std::aligned_alloc`（可分页） | `aclrtMemcpy` | 同步 | 基线 |
| v2 | `aclrtMallocHost`（锁页） | `aclrtMemcpy` | 同步 | 锁页内存 |
| v3 | `std::aligned_alloc`（可分页） | `aclrtMemcpyAsync` | **退化为同步** | 异步接口的前提条件 |
| v4 | `aclrtMallocHost`（锁页） | `aclrtMemcpyAsync` | **真正异步下发** | 真正的异步复制 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">Host 内存</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">复制接口</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">接口的实际行为</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">引入的新概念</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>std::aligned_alloc</code>（可分页）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpy</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">基线</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMallocHost</code>（锁页）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpy</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">锁页内存</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>std::aligned_alloc</code>（可分页）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpyAsync</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>退化为同步</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步接口的前提条件</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMallocHost</code>（锁页）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpyAsync</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>真正异步下发</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">真正的异步复制</td>
</tr>
</tbody>
</table>

之所以要做成全组合而不是三个递进版本，是因为两个因素并不独立：由 §3.2 可知，异步接口以锁页内存为前提。全组合能把这层关系直接测出来——**四个格子里只有一个是真正异步的**，另外三个在行为上都是同步。若只做递进式的 v1 → v2 → v3，就看不到 v3（可分页 + 异步）这个反例，也就无法说明异步不是由接口名决定的。

每个版本都在两个方向上测量：

- **H2D**（Host to Device）：主机内存 → 设备内存，对应模型推理中送入输入数据
- **D2H**（Device to Host）：设备内存 → 主机内存，对应取回推理结果

传输规模按 2 的幂从 1 KB 扫到 128 MB，共 18 个测量点。每个版本先用**该版本自己的复制路径**做一次往返校验，确认数据搬运正确，再开始计时。


## 7. 程序实现

本实验只有**一个源文件** `src_memory/acl_memory.cpp`，分五段写入：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 顺序 | 命令 | 内容 |
| --- | --- | --- |
| 1 | `%%writefile` | 头文件、常量与错误检查宏 |
| 2 | `-a` 追加 | 计时工具与重复次数策略 |
| 3 | `-a` 追加 | 主机内存的两种申请方式与复制接口的两种调用方式 |
| 4 | `-a` 追加 | 往返校验与单点测量 |
| 5 | `-a` 追加 | 版本循环与 `main` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">顺序</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">命令</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">1</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>%%writefile</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">头文件、常量与错误检查宏</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">2</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">计时工具与重复次数策略</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">3</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机内存的两种申请方式与复制接口的两种调用方式</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">往返校验与单点测量</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">5</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>-a</code> 追加</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">版本循环与 <code>main</code></td>
</tr>
</tbody>
</table>

> ⚠️ 第一个代码单元格使用 `%%writefile`（**覆盖创建**），其后四个使用 `%%writefile -a`（**追加**）。修改代码后需要从 7.1 开始按顺序重新执行，否则文件内容会重复或缺失。

### 7.1 头文件、常量与错误检查宏

`ACL_CHECK` 宏是本章统一的错误检查写法。三个常量控制扫描范围：最小与最大传输规模的以 2 为底的对数，以及可分页缓冲区的对齐字节数。


In [ ]:
%%writefile src_memory/acl_memory.cpp
/**
 * Parallel Computing, Chapter 7, Lab 2: Host-Device Memory and Data Transfer
 *
 * This program measures the effective bandwidth between host memory and
 * device memory as a function of the transfer size. Four versions form a
 * two-factor design: host memory kind (pageable or page-locked) crossed with
 * the copy API (synchronous or asynchronous). Both directions are measured.
 *
 * The measured points are meant to be fitted to T = alpha + S / beta, where
 * alpha is the fixed cost of one transfer and beta is the asymptotic
 * bandwidth.
 */
#include <cstdint>  // int32_t, uint8_t
#include <cstdio>   // std::printf, std::fprintf
#include <cstdlib>  // std::aligned_alloc, std::free
#include <cstring>  // std::memset, std::memcmp
#include <ctime>    // clock_gettime, timespec

#include "acl/acl.h"  // Runtime resource management APIs

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kMinLog2Bytes = 10;  // 1 KB
constexpr int kMaxLog2Bytes = 27;  // 128 MB
constexpr int kWarmupRuns = 3;
// aclrtMemcpyAsync requires both the source and the destination address to be
// 64-byte aligned. aclrtMalloc and aclrtMallocHost guarantee this, plain
// malloc does not, so pageable buffers are aligned explicitly.
constexpr size_t kHostAlign = 64;

}  // namespace

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 7.2 计时工具与重复次数策略

计时用 `CLOCK_MONOTONIC` 单调时钟：它自系统启动起单调递增，不受 NTP 校时或手动改时间的影响，因此测量时间间隔必须用它，不能用 `CLOCK_REALTIME`。

重复次数随传输规模递减。小规模单次耗时只有几微秒，与时钟本身的分辨率同量级，必须多次重复才能压住抖动；大规模单次耗时上十毫秒，重复太多次只是浪费时间。这里按规模分三档，使每个测量点的总耗时大致相当。


In [ ]:
%%writefile -a src_memory/acl_memory.cpp

// Returns a monotonic timestamp in milliseconds. CLOCK_MONOTONIC increases
// steadily since system start and is unaffected by wall-clock adjustments
// such as NTP, which makes it the right clock for measuring intervals.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Picks the repeat count from the transfer size so that every measurement
// point takes roughly the same total time. Small transfers need many
// repetitions to average out jitter; large ones do not.
int RepeatFor(size_t bytes) {
  if (bytes <= (static_cast<size_t>(1) << 20)) {
    return 100;
  }
  if (bytes <= (static_cast<size_t>(1) << 24)) {
    return 20;
  }
  return 5;
}


### 7.3 两种主机内存与两种复制方式

这一段把版本这个概念落成两个枚举和四个函数。

`AllocHost` 中的可分页分支用 `std::aligned_alloc(64, bytes)` 而不是 `malloc`，原因见 §2.3。注意 `std::aligned_alloc` 要求申请的字节数是对齐值的整数倍——本实验的规模都是 2 的幂且不小于 1 KB，天然满足。

`CopyOnce` 中的异步分支在下发之后立即同步。这不是为了让异步退化成同步，而是因为**本实验测的是单次传输的耗时**，必须等它真正完成才能停计时。异步接口的价值要在传输与计算重叠的场景中才体现出来，本实验不做重叠。


In [ ]:
%%writefile -a src_memory/acl_memory.cpp

enum class HostMemKind { kPageable, kPinned };
enum class CopyApi { kSync, kAsync };

const char* HostKindName(HostMemKind kind) {
  return (kind == HostMemKind::kPinned) ? "pinned" : "pageable";
}

const char* CopyApiName(CopyApi api) {
  return (api == CopyApi::kAsync) ? "async" : "sync";
}

// Allocates a host buffer of the requested kind.
int AllocHost(HostMemKind kind, size_t bytes, void** ptr) {
  if (kind == HostMemKind::kPinned) {
    ACL_CHECK(aclrtMallocHost(ptr, bytes));
    return ACL_SUCCESS;
  }
  *ptr = std::aligned_alloc(kHostAlign, bytes);
  if (*ptr == nullptr) {
    std::fprintf(stderr,
                 "[ERR] api=std::aligned_alloc code=- msg=returned nullptr\n");
    return ACL_ERROR_INVALID_PARAM;
  }
  return ACL_SUCCESS;
}

// Releases a host buffer. Page-locked memory must be released by
// aclrtFreeHost; the two allocators are not interchangeable.
int FreeHost(HostMemKind kind, void* ptr) {
  if (kind == HostMemKind::kPinned) {
    ACL_CHECK(aclrtFreeHost(ptr));
    return ACL_SUCCESS;
  }
  std::free(ptr);
  return ACL_SUCCESS;
}

// Performs one copy. The asynchronous branch synchronizes right away because
// this lab measures the duration of a single transfer; overlapping transfers
// with computation is the subject of Lab 3.
int CopyOnce(CopyApi api, void* dst, size_t dst_max, const void* src,
             size_t count, aclrtMemcpyKind kind, aclrtStream stream) {
  if (api == CopyApi::kSync) {
    ACL_CHECK(aclrtMemcpy(dst, dst_max, src, count, kind));
    return ACL_SUCCESS;
  }
  ACL_CHECK(aclrtMemcpyAsync(dst, dst_max, src, count, kind, stream));
  ACL_CHECK(aclrtSynchronizeStream(stream));
  return ACL_SUCCESS;
}


### 7.4 往返校验与单点测量

`VerifyRoundTrip` 写入一段有规律的字节序列，搬到设备再搬回来，逐字节比对。这里用 `std::memcmp` 做**位级比对**是正确的：内存复制不做任何运算，一字节不差是它的定义。浮点计算常用的相对误差阈值在这里反而是错的——它会掩盖单个字节的错误。

**校验走的是该版本自己的复制路径**：函数接收 `api` 与 `stream` 两个参数，两次搬运都经由 `CopyOnce` 发起，与随后计时用的是同一条路。若校验一律用同步接口，v3 与 v4 的异步路径在计时之前就从未被功能验证过，随后测出的性能数据便是 §6 所说的未经校验的数据。这一点在异步接口上尤其要紧：`aclrtMemcpyAsync` 之后若漏掉 `aclrtSynchronizeStream`，读回的就是尚未完成的数据。

`MeasureOne` 完成一个测量点：先预热，再计时重复，最后按 §5.3 的单位约定折算带宽并打印一行 `[PERF]` 记录。记录行采用 `key=value` 空格分隔的形式，便于 Python 端用一行正则解析。


In [ ]:
%%writefile -a src_memory/acl_memory.cpp

// Verifies a host -> device -> host round trip byte by byte, using the copy
// path of the version under test: the two transfers go through CopyOnce with
// this version's api and stream, so the asynchronous path is functionally
// verified before it is timed. A memory copy must reproduce every byte
// exactly, and no arithmetic is involved, so a relative-error tolerance
// would be the wrong tool here.
int VerifyRoundTrip(const char* ver, CopyApi api, size_t bytes, void* host_src,
                    void* host_dst, void* device_ptr, aclrtStream stream) {
  uint8_t* src = static_cast<uint8_t*>(host_src);
  for (size_t i = 0; i < bytes; ++i) {
    src[i] = static_cast<uint8_t>((i * 31u + 7u) & 0xFFu);
  }
  std::memset(host_dst, 0, bytes);
  ACL_CHECK(CopyOnce(api, device_ptr, bytes, host_src, bytes,
                     ACL_MEMCPY_HOST_TO_DEVICE, stream));
  ACL_CHECK(CopyOnce(api, host_dst, bytes, device_ptr, bytes,
                     ACL_MEMCPY_DEVICE_TO_HOST, stream));
  const bool ok = (std::memcmp(host_src, host_dst, bytes) == 0);
  std::printf("[VERIFY] ver=%s api=%s bytes=%zu result=%s\n", ver,
              CopyApiName(api), bytes, ok ? "PASS" : "FAIL");
  return ok ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}

// Measures one point of the scan and prints a machine-readable record.
int MeasureOne(const char* ver, HostMemKind host_kind, CopyApi api,
               bool host_to_device, size_t bytes, void* host_ptr,
               void* device_ptr, aclrtStream stream) {
  void* dst = host_to_device ? device_ptr : host_ptr;
  const void* src = host_to_device ? host_ptr : device_ptr;
  const aclrtMemcpyKind kind =
      host_to_device ? ACL_MEMCPY_HOST_TO_DEVICE : ACL_MEMCPY_DEVICE_TO_HOST;

  for (int i = 0; i < kWarmupRuns; ++i) {
    ACL_CHECK(CopyOnce(api, dst, bytes, src, bytes, kind, stream));
  }

  const int repeat = RepeatFor(bytes);
  const double t0 = GetTimeMs();
  for (int i = 0; i < repeat; ++i) {
    ACL_CHECK(CopyOnce(api, dst, bytes, src, bytes, kind, stream));
  }
  const double ms = (GetTimeMs() - t0) / repeat;

  // Capacity units are binary throughout this chapter: 1 GB = 2^30 bytes.
  constexpr double kBytesPerGb = 1024.0 * 1024.0 * 1024.0;
  const double gbps = static_cast<double>(bytes) / (ms * 1.0e-3) / kBytesPerGb;
  std::printf(
      "[PERF] ver=%s host=%s api=%s dir=%s bytes=%zu repeat=%d ms=%.6f "
      "gbps=%.4f\n",
      ver, HostKindName(host_kind), CopyApiName(api),
      host_to_device ? "h2d" : "d2h", bytes, repeat, ms, gbps);
  return ACL_SUCCESS;
}


### 7.5 版本循环与主程序

`RunOneVersion` 按最大规模申请一次内存，供该版本的全部 18 个规模共用，避免把内存申请的开销混进传输耗时。它分成两个函数：`RunOneVersionBody` 负责申请、校验与测量，任何一步失败都立即返回；`RunOneVersion` 负责在它返回之后，按申请的逆序释放三块内存。这样即使中途失败，已经申请到的资源也不会泄漏。

`aclrtMalloc` 申请到的 Device 内存不会被初始化，因此紧接着调用一次 `aclrtMemset` 清零。它同时充当 Device 侧的首次触碰，也使 2 MB 及以上规模的 D2H 测量不再从未初始化的内存中读数据。

申请之后立即对两块主机缓冲区各做一次 `memset`，这一步称为**首次触碰**。它针对的是可分页缓冲区：操作系统采用惰性分配，`aligned_alloc` 返回时物理页尚未真正映射，第一次写入才会触发缺页中断。不先触碰一遍，首个测量点就会同时付出缺页的代价。锁页内存在申请时已经完成虚拟页与物理页的绑定，这一步对它是多余的，但四个版本走同一段代码可以保证测量条件一致。

`main` 中四个版本用一张配置表驱动，逐个执行。


In [ ]:
%%writefile -a src_memory/acl_memory.cpp

// Runs the full size scan for one version, in both directions. Buffers are
// allocated here but released by the caller, so an early return on failure
// cannot leak them.
int RunOneVersionBody(const char* ver, HostMemKind host_kind, CopyApi api,
                      aclrtStream stream, size_t max_bytes, void** host_a,
                      void** host_b, void** device_ptr) {
  ACL_CHECK(AllocHost(host_kind, max_bytes, host_a));
  ACL_CHECK(AllocHost(host_kind, max_bytes, host_b));
  ACL_CHECK(aclrtMalloc(device_ptr, max_bytes, ACL_MEM_MALLOC_HUGE_FIRST));
  // aclrtMalloc does not initialise the memory it returns, so clear it once.
  // This also acts as the device-side counterpart of the first touch below.
  ACL_CHECK(aclrtMemset(*device_ptr, max_bytes, 0, max_bytes));

  // First touch: writing to every page forces the operating system to back
  // the buffers with physical pages before timing starts. Without it the
  // first measurement would also pay for page faults. Page-locked memory is
  // already backed on allocation; all four versions share this code so that
  // the measurement conditions stay identical.
  std::memset(*host_a, 0xA5, max_bytes);
  std::memset(*host_b, 0x5A, max_bytes);

  ACL_CHECK(VerifyRoundTrip(ver, api, static_cast<size_t>(1) << 20, *host_a,
                            *host_b, *device_ptr, stream));

  for (int k = kMinLog2Bytes; k <= kMaxLog2Bytes; ++k) {
    const size_t bytes = static_cast<size_t>(1) << k;
    ACL_CHECK(MeasureOne(ver, host_kind, api, true, bytes, *host_a,
                         *device_ptr, stream));
    ACL_CHECK(MeasureOne(ver, host_kind, api, false, bytes, *host_b,
                         *device_ptr, stream));
  }
  return ACL_SUCCESS;
}

// Allocates, runs and releases. Releasing here rather than at the end of the
// body means that a failure anywhere inside the body still frees whatever had
// already been allocated, in the reverse order of allocation.
int RunOneVersion(const char* ver, HostMemKind host_kind, CopyApi api,
                  aclrtStream stream) {
  const size_t max_bytes = static_cast<size_t>(1) << kMaxLog2Bytes;
  void* host_a = nullptr;
  void* host_b = nullptr;
  void* device_ptr = nullptr;
  const int ret = RunOneVersionBody(ver, host_kind, api, stream, max_bytes,
                                    &host_a, &host_b, &device_ptr);
  if (device_ptr != nullptr) {
    (void)aclrtFree(device_ptr);
  }
  if (host_b != nullptr) {
    (void)FreeHost(host_kind, host_b);
  }
  if (host_a != nullptr) {
    (void)FreeHost(host_kind, host_a);
  }
  return ret;
}

namespace {

struct VersionConfig {
  const char* ver;
  HostMemKind host_kind;
  CopyApi api;
};

constexpr VersionConfig kVersions[] = {
    {"v1", HostMemKind::kPageable, CopyApi::kSync},
    {"v2", HostMemKind::kPinned, CopyApi::kSync},
    {"v3", HostMemKind::kPageable, CopyApi::kAsync},
    {"v4", HostMemKind::kPinned, CopyApi::kAsync},
};

}  // namespace

int main() {
  std::printf("[INFO] acl_memory start\n");

  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));

  // On Atlas A2 and A3 products ACL_DEVICE is not supported, so this always
  // reports ACL_HOST. It is printed to confirm the expected run mode.
  aclrtRunMode run_mode = ACL_HOST;
  ACL_CHECK(aclrtGetRunMode(&run_mode));
  std::printf("[INFO] run_mode = %s\n",
              (run_mode == ACL_HOST) ? "ACL_HOST" : "ACL_DEVICE");

  aclrtContext context = nullptr;
  ACL_CHECK(aclrtCreateContext(&context, kDeviceId));
  aclrtStream stream = nullptr;
  ACL_CHECK(aclrtCreateStream(&stream));

  int ret = ACL_SUCCESS;
  for (const VersionConfig& cfg : kVersions) {
    ret = RunOneVersion(cfg.ver, cfg.host_kind, cfg.api, stream);
    if (ret != ACL_SUCCESS) {
      break;
    }
  }

  ACL_CHECK(aclrtDestroyStream(stream));
  ACL_CHECK(aclrtDestroyContext(context));
  ACL_CHECK(aclrtResetDevice(kDeviceId));
  ACL_CHECK(aclFinalize());

  std::printf("[RESULT] %s\n", (ret == ACL_SUCCESS) ? "PASS" : "FAILED");
  std::printf("[INFO] acl_memory finished\n");
  return (ret == ACL_SUCCESS) ? 0 : ret;
}


## 8. 编译与运行

一条 `g++` 命令即可完成编译。程序只包含 `acl/acl.h`，因此只链接 Runtime 库。


In [ ]:
import os, subprocess

SRC = "src_memory/acl_memory.cpp"
EXE = "src_memory/acl_memory"

# g++ [源文件] -I[头文件目录] [库目录] [库名] -o [可执行文件]
cmd = (
    ["g++", SRC, "-std=c++17", "-O2", "-Wall"]
    + ["-I" + os.environ["ASCEND_HOME_PATH"] + "/include"]
    + ACL_LIBDIRS
    + ACL_RT_LIB
    + ["-o", EXE]
)
print("$ " + " ".join(cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
msg = (proc.stdout + proc.stderr).strip()
if msg:
    print(msg)

print(
    "✅ 编译成功"
    if proc.returncode == 0
    else "❌ 编译失败（返回码 %d）" % proc.returncode
)


一次运行即可输出四个版本、两个方向、18 个规模共 144 条性能记录，以及 4 条校验记录。全程约需一到两分钟，其中大规模测量占大部分时间。

输出较长，下面只打印首尾各若干行，完整内容保存在变量 `out_main` 中供后续解析。


In [ ]:
import subprocess

proc = subprocess.run(
    ["./src_memory/acl_memory"], capture_output=True, text=True, timeout=1800
)
out_main = proc.stdout
if proc.returncode != 0:
    print("返回码:", proc.returncode)
    print(proc.stderr)

lines = out_main.splitlines()
print("\n".join(lines[:12]))
print("    ... 共 %d 行 ..." % len(lines))
print("\n".join(lines[-6:]))


## 9. 解析输出

下面把 `[PERF]` 与 `[VERIFY]` 记录行解析成字典列表，后面的表格、图表与 §11 的求解都基于它们。解析规则：`key=value` 空格分隔，数值字段转成 `float`，其余保持字符串。


In [ ]:
import re


def parse_perf(text):
    # 把所有 [PERF] 行解析为 dict 列表
    numeric = {"bytes", "repeat", "ms", "gbps"}
    rows = []
    for line in text.splitlines():
        if line.startswith("[PERF]"):
            d = {}
            for kv in line.split()[1:]:
                k, v = kv.split("=", 1)
                d[k] = float(v) if k in numeric else v
            rows.append(d)
    return rows


def parse_verify(text):
    return {
        m.group(1): m.group(2)
        for m in re.finditer(r"\[VERIFY\] ver=(\S+).*?result=(\S+)", text)
    }


rows_main = parse_perf(out_main)
chk_main = parse_verify(out_main)
VERS = ["v1", "v2", "v3", "v4"]
LABEL = {
    "v1": "v1 pageable + sync",
    "v2": "v2 pinned   + sync",
    "v3": "v3 pageable + async",
    "v4": "v4 pinned   + async",
}


def series(ver, direction):
    # 取某个版本某个方向的 (bytes, ms, gbps) 序列，按规模升序
    rows = [r for r in rows_main if r["ver"] == ver and r["dir"] == direction]
    rows.sort(key=lambda r: r["bytes"])
    return rows


print("记录行 %d 条，校验结果：%s" % (len(rows_main), chk_main))
print()

# 八列 = 四个版本 × 两个方向，与 §11 的八行一一对应，
# 便于把解出的常数拿回到实测曲线上核对。
COLUMNS = [(ver, direction) for direction in ("h2d", "d2h") for ver in VERS]
table_data = {key: series(*key) for key in COLUMNS}

row_fmt = "%-9s" + " %9.2f" * len(COLUMNS)
header = ["规模"] + ["%s %s" % (v, d.upper()) for v, d in COLUMNS]
print(("%-9s" + " %9s" * len(COLUMNS)) % tuple(header))
for i, ref in enumerate(table_data[COLUMNS[0]]):
    size = int(ref["bytes"])
    name = "%d KB" % (size // 1024) if size < 1024**2 else "%d MB" % (size // 1024**2)
    values = [table_data[key][i]["gbps"] for key in COLUMNS]
    print(row_fmt % tuple([name] + values))
print()
print("（单位 GB/s，1 GB = 2^30 字节）")


## 10. 结果可视化：带宽曲线

下面绘制两张图，分别对应两个传输方向。每张图上有四条曲线，对应四个版本。

按 $T = \alpha + S/\beta$ 推断，带宽曲线应当分为三段：小规模段带宽随规模近似线性上升（此时时间几乎恒为 $\alpha$，带宽 $\approx S/\alpha$）、中间的过渡段、大规模段趋于水平渐近线 $\beta$。

读图时有三处需要注意：

1. **v1 与 v3 的曲线会几乎完全重叠**，灰色的 v1 会被橙色的 v3 覆盖。这不是漏画，恰恰是本实验要观察的现象，§12 ③ 会解释。
2. 四条曲线是否都呈现上面那个三段形状，本身就是一个需要检验的问题。**若某条曲线在中途出现台阶或回落，说明该配置在那个规模上跨过了一条边界**，应当在 §12 中单独讨论。
3. **本图的纵轴是线性刻度**，量程要覆盖到几十 GB/s，因此小规模段的四条曲线全被压在接近零的位置，看上去几乎重合。这是刻度造成的，不是实测结果——请在 §9 的表格中查 16 KB 那一行，读出 v4 的带宽是 v2 的百分之几，再回到图上看这个差别能否分辨出来。**要比较小规模段的差别，应当看 §11 的双对数图**，那里四个版本的水平段高低分明。

半带宽规模 $S_{1/2}$ 的参考线不在本图中，把它画上去是动手练习第 1 题的内容。


In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

C_V1, C_V2, C_V3, C_V4 = "#9AA5B1", "#3B6FE0", "#E07A3B", "#2E9E6B"
COLOR = {"v1": C_V1, "v2": C_V2, "v3": C_V3, "v4": C_V4}
MARKER = {"v1": "o", "v2": "s", "v3": "^", "v4": "D"}

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4), dpi=120, sharey=True)
for ax, direction, title in zip(
    axes, ("h2d", "d2h"), ("Host to Device", "Device to Host")
):
    for ver in VERS:
        rows = series(ver, direction)
        ax.plot(
            [r["bytes"] for r in rows],
            [r["gbps"] for r in rows],
            marker=MARKER[ver],
            ms=4,
            lw=1.8,
            color=COLOR[ver],
            label=LABEL[ver],
        )
    ax.set_xscale("log", base=2)
    ax.set_xlabel("Transfer size (bytes)")
    ax.set_title("Lab 2: %s" % title)
    ax.grid(alpha=0.3, which="both")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel("Effective bandwidth (GB/s, 2^30)")
axes[0].legend(frameon=False, loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


## 11. 从曲线两端读出 $\alpha$ 与 $\beta$

§9 的表格里已经含有 $\alpha$ 与 $\beta$，只是需要从两端分别把它们读出来。

**$\alpha$ 在小规模那一端。** 传输量足够小时 $S/\beta$ 可以忽略，实测耗时就近似等于 $\alpha$。表中最小的那几个规模，耗时几乎不随规模变化，这个几乎不变的值就是 $\alpha$。

**$\beta$ 在大规模那一端。** 传输量足够大时 $\alpha$ 可以忽略；把 $\alpha$ 从耗时中扣除，$S/(T-\alpha)$ 就是 $\beta$。

难点只有一个：足够小与足够大的界线画在哪里。这条界线**不能写成一个常数**——它随平台变化，同一个阈值在一台机器上落在小规模段的正中间，在另一台上已经越过了过渡段。界线由模型本身给出：

- 小规模段的条件是 $S/\beta \ll \alpha$，即 $S \ll \alpha\beta = S_{1/2}$；
- 大规模段的条件是 $\alpha \ll S/\beta$，即 $S \gg S_{1/2}$。

两个条件都用 $S_{1/2}$ 表示，而 $S_{1/2}$ 又要由 $\alpha$ 与 $\beta$ 算出。因此下面的代码先用两端各两个点粗估，得到 $S_{1/2}$ 的初值，再按 $S \le S_{1/2}/10$ 与 $S \ge 10\,S_{1/2}$ 重新划段、重新求解，迭代到收敛为止。把远大于取作一个数量级，同时也给出了误差上界：在 $S = S_{1/2}/10$ 处模型给出 $T = \alpha(1 + 1/10)$，因此这样读出的 $\alpha$ 至多偏高一成；大规模段那一侧已经把 $\alpha$ 显式扣除，$\beta$ 不带这项偏差。

还有一条约束与模型无关：**当带宽在量程中间骤降时，大规模段必须整体落在骤降之后**，否则解出的 $\beta$ 是跨越两个区间的混合值，既不是骤降之前的带宽，也不是骤降之后的带宽。带宽随规模上升是模型预期的，带宽在某个规模上明显下降则不是，因此代码把相邻两点的带宽跌幅超过四分之一认定为一次骤降，并要求大规模段整体落在最后一次骤降之后。

最终采用的两个分段点、各段实际用到的点数、以及大规模段下界的来源都会打印出来，**请在读 $\alpha$ 与 $\beta$ 之前先看这三列**。

读出的两个常数是否代表整条路径，由两条自检判定，代码逐行给出结论：

1. **小规模段是否真的平坦。** 模型说这一段的耗时恒为 $\alpha$。代码取该段耗时的最大值与最小值之比。这个比值的容许上界由模型自己给出：该段的上界取在 $S_{1/2}/10$，模型在那里给出 $T = \alpha(1 + 1/10)$，因此极差不应超过 1.10；留出测量抖动的余量，代码以 1.25 为界。比值在界内，读出的就是 $\alpha$ 本身；超出界外，说明这一段仍在爬升，读出的 $\alpha$ 只是一个上界。
2. **量程中间是否存在带宽骤降。** 骤降说明这条路径在某个规模上换了一种工作方式，一组 $(\alpha, \beta)$ 描述不了整个量程，骤降两侧各有一组。

两条都通过时，$(\alpha, \beta, S_{1/2})$ 可以代表这条路径的全量程；任何一条不通过，引用这三个数之前都要先看清它们只在哪一段上成立。


In [ ]:
# 分段点不写成常数，而由模型自身的判据推出（推导见 §11）：
#   小规模段要求 S ≪ αβ = S½ ，本实验取 S ≤ S½ / kSegRatio
#   大规模段要求 S ≫ S½      ，本实验取 S ≥ S½ × kSegRatio
# α、β 与 S½ 互相依赖，因此先用两端各两个点粗估，再迭代到收敛。
kSegRatio = 10.0  # 判据中的远大于取一个数量级
kFitIters = 6  # 迭代次数上限
kMinSegPts = 3  # 每段至少保留的点数
kStepDrop = 0.25  # 相邻两点带宽跌幅超过它，就认定量程中间存在一次骤降
# 模型自身预言小规模段的极差不超过 1.10：该段上界取 S½/10，在那里 T = α(1 + 1/10)。
# 留出测量抖动的余量，取 1.25 为界。
kFlatTol = 0.25
GIB = 1024.0**3


def last_step(gbps):
    # 返回最后一次带宽显著下降之后的下标；没有下降时返回 0。
    # 带宽随规模上升是模型预期的；带宽在某个规模上明显下降，说明量程中间
    # 换了一种工作方式，跨过它取点解出的 β 是两个区间的混合值。
    start = 0
    for i in range(1, len(gbps)):
        if gbps[i] < (1.0 - kStepDrop) * gbps[i - 1]:
            start = i
    return start


def solve_alpha_beta(rows):
    x = np.array([r["bytes"] for r in rows], dtype=float)
    y = np.array([r["ms"] for r in rows], dtype=float)
    gbps = np.array([r["gbps"] for r in rows], dtype=float)
    idx = np.arange(len(x))
    lo = last_step(gbps)  # 大规模段必须整体落在最后一次骤降之后

    # 小规模段直接读出 α，大规模段扣除 α 之后解出 β。
    # 初值：最小两点估 α，最大两点估 β。
    alpha_ms = float(np.mean(y[:2]))
    bytes_per_s = float(np.mean(x[-2:] / (y[-2:] - alpha_ms)) * 1.0e3)
    small = idx < kMinSegPts
    large = idx >= max(lo, len(x) - kMinSegPts)
    small_by = large_by = "点数下限"
    for _ in range(kFitIters):
        s_half = alpha_ms * 1.0e-3 * bytes_per_s

        small = x <= s_half / kSegRatio
        small_by = "S½/10 判据"
        if int(small.sum()) < kMinSegPts:  # 保底：至少留下最小的几个点
            small, small_by = idx < kMinSegPts, "点数下限"

        by_crit = x >= s_half * kSegRatio
        large = by_crit & (idx >= lo)
        if int(large.sum()) < kMinSegPts:  # 保底：至少留下最大的几个点
            large, large_by = idx >= max(lo, len(x) - kMinSegPts), "点数下限"
        elif int(large.sum()) < int(by_crit.sum()):
            large_by = "骤降之后"  # 骤降使下界上移
        else:
            large_by = "10·S½ 判据"

        alpha_new = float(np.mean(y[small]))
        beta_new = float(
            np.mean(x[large] / np.maximum(y[large] - alpha_new, 1.0e-9)) * 1.0e3
        )
        done = abs(alpha_new - alpha_ms) <= 0.01 * alpha_ms and (
            abs(beta_new - bytes_per_s) <= 0.01 * bytes_per_s
        )
        alpha_ms, bytes_per_s = alpha_new, beta_new
        if done:
            break

    # 自检一：小规模段是否真的平坦。模型说这一段耗时恒为 α。
    ys = y[small]
    flat = float(ys.max() / ys.min()) if ys.min() > 0 else float("inf")

    return {
        "alpha_us": alpha_ms * 1000.0,
        "beta_gbps": bytes_per_s / GIB,
        "s_half": alpha_ms * 1.0e-3 * bytes_per_s,
        "flat_ratio": flat,
        "small_max": float(x[small].max()),
        "n_small": int(small.sum()),
        "small_by": small_by,
        "large_min": float(x[large].min()),
        "n_large": int(large.sum()),
        "large_by": large_by,
        "stepped": bool(lo > 0),          # 自检二：量程中间是否有带宽骤降
        "step_at": float(x[lo]) if lo > 0 else -1.0,
    }


def format_size(nbytes):
    if nbytes < 0:
        return "n/a"
    if nbytes < 1024**2:
        return "%.1f KB" % (nbytes / 1024)
    return "%.2f MB" % (nbytes / 1024**2)


hdr = ("版本", "方向", "α (µs)", "β (GB/s)", "S½", "小段极差", "带宽骤降")
print("%-22s %-5s %10s %10s %11s %9s  %s" % hdr)
print("-" * 88)
fit_result = {}
for ver in VERS:
    for direction in ("h2d", "d2h"):
        f = solve_alpha_beta(series(ver, direction))
        fit_result[(ver, direction)] = f
        print(
            "%-22s %-5s %10.2f %10.2f %11s %8.2f×  %s"
            % (
                LABEL[ver],
                direction,
                f["alpha_us"],
                f["beta_gbps"],
                format_size(f["s_half"]),
                f["flat_ratio"],
                ("有，起于 " + format_size(f["step_at"])) if f["stepped"] else "无",
            )
        )

print()
print("上表用到的分段点由 α、β 迭代推出，不是写死的常数（判据见 §11）：")
print(
    "%-22s %-5s %11s %5s %11s %5s  %s"
    % ("版本", "方向", "小段上界", "点数", "大段下界", "点数", "大段下界的来源")
)
print("-" * 88)
for ver in VERS:
    for direction in ("h2d", "d2h"):
        f = fit_result[(ver, direction)]
        print(
            "%-22s %-5s %11s %5d %11s %5d  %s"
            % (
                LABEL[ver],
                direction,
                format_size(f["small_max"]),
                f["n_small"],
                format_size(f["large_min"]),
                f["n_large"],
                f["large_by"],
            )
        )


def check_row(f):
    # 两条自检，都直接来自模型本身
    bad = []
    if f["flat_ratio"] > 1.0 + kFlatTol:
        bad.append("小规模段仍在爬升，读出的 α 只是上界")
    if f["stepped"]:
        bad.append("量程中间有带宽骤降，一组 (α, β) 描述不了全量程")
    return bad


print()
warn = [(k, check_row(f)) for k, f in fit_result.items() if check_row(f)]
if warn:
    print("⚠️  以下组合上，(α, β) 只在一段量程内成立：")
    for (ver, direction), bad in warn:
        print("    %-22s %-5s %s" % (LABEL[ver], direction, "；".join(bad)))
    print("    引用这几行的 α、β、S½ 之前，请先读 §12 ① 与 ②。")
else:
    print("✅ 八个组合两条自检全部通过，(α, β, S½) 在本机整个量程上成立。")


下图把结果画出来：横轴为传输量、纵轴为耗时，散点是实测值，虚线是按解出的 $\alpha$ 与 $\beta$ 画出的模型曲线。两条坐标轴都用对数刻度，可以同时看清小规模段的水平段（时间恒为 $\alpha$）与大规模段斜率为 1 的上升段（时间正比于传输量）。

这张图比带宽曲线更适合读出 $\alpha$：**小规模段那条水平段的高度就是 $\alpha$**，四个版本的高低可以直接读出。同时也要留意哪几条虚线没有穿过对应的散点——虚线偏离散点的地方，就是模型不成立的地方。


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.6), dpi=120)
# 模型曲线的量程随实测数据走，改了 §7.5 的规模循环之后这里无需同步修改
all_bytes = [r["bytes"] for r in rows_main]
x_fit = np.logspace(np.log2(min(all_bytes)), np.log2(max(all_bytes)), 200, base=2)
for ver in VERS:
    rows = series(ver, "h2d")
    f = fit_result[(ver, "h2d")]
    a, b = f["alpha_us"], f["beta_gbps"]
    ax.plot(
        [r["bytes"] for r in rows],
        [r["ms"] for r in rows],
        MARKER[ver],
        ms=4,
        color=COLOR[ver],
        label=LABEL[ver],
    )
    ax.plot(
        x_fit,
        a * 1.0e-3 + x_fit / (b * 1024.0**3) * 1.0e3,
        lw=1.2,
        ls="--",
        color=COLOR[ver],
        alpha=0.7,
    )
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Transfer size (bytes)")
ax.set_ylabel("Transfer time (ms)")
ax.set_title("Lab 2: H2D transfer time vs model T = alpha + S / beta")
ax.grid(alpha=0.3, which="both")
ax.legend(frameon=False, loc="upper left", fontsize=9)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


## 12. 结果分析

> 本节**不引用任何一次运行的具体数值**。需要数值的地方，请读本机 §9 的表格与 §11 的输出。同一段代码在不同的硬件规格、总线类型、CANN 版本与系统负载下，给出的不只是量级不同的数字——**其中一部分定性结论也会随之改变**。凡是这类结论，下面都写成了分支判据，并注明各分支在真实硬件上都会出现；判断本机落在哪一支，是本节要求学生自己完成的工作。

**① 先判断 $T = \alpha + S/\beta$ 在哪几个版本上成立。**

模型预言的带宽曲线分三段：小规模段传输时间恒为 $\alpha$，因而带宽 $S/\alpha$ 随规模线性上升；大规模段固定开销可以忽略，带宽趋于 $\beta$；中间是过渡段。这个形状是否成立，不能靠看图，要靠 §11 的两条自检。

请对照本机 §11 的输出回答三个问题：

- 八个组合中，哪几行的小段极差明显大于 1？哪几行有带宽骤降？
- 有骤降的行，是否恰好是同一类版本（可分页或锁页、同步或异步）？
- 骤降起于哪一个规模？

两条自检对应两种不同的局面，**下面三种情形在真实硬件上都会出现**：

- **出现带宽骤降**：某个版本的带宽在某个规模上跌到低位段上。此时模型在整个量程上不成立，骤降两侧各有一组 $(\alpha, \beta)$。
- **小规模段仍在爬升**：曲线没有骤降，但最小的几个点耗时并不相等。此时读出的 $\alpha$ 仍可使用，但它是一个上界。
- **八行全部通过**：四个版本都可以用一组 $(\alpha, \beta)$ 描述整个量程。

落到哪一种情形，不是测量误差，而是本机这条传输路径的性质。**同一段代码在两台参考机上落到了不同的情形**——这一点本身就是本实验最值得记住的结果之一。

**② 可分页与锁页的差别：先看有没有边界，再看增益有多大。**

先回到本机 §9 的表格，回答一个二选一的问题：v1 与 v3 的带宽是随规模单调上升到一条渐近线，还是在某个规模上骤降之后维持在一个低位段上？

- **若出现骤降**：请记下下降发生在哪两个相邻测量点之间，并对照 v2 在同一区间的表现。若 v2 在同一区间继续上升，这个下降就只能归因于可分页这一条路径——四个版本除主机内存类型与复制接口之外完全相同。此时 $T = \alpha + S/\beta$ 对可分页传输只能分段使用，**跨越边界外推得到的数值不可信**。
- **若没有骤降**：说明本机上可分页与锁页走的是同一量级的通路，四个版本都可以用单段模型描述，② 的后半段仍然要问，但答案会小得多。

关于这条路径，《应用开发指南》写明的只有一件事：**可分页内存的数据先复制到 Runtime 内部的锁页缓冲区，再经 DMA 通道传输到 Device。** 这一步中转复制由 CPU 完成，代价正比于数据量。文档没有给出这块缓冲区的容量，本实验的测量点按 2 的整数次幂取值、间隔过粗，也不足以定位它。因此本实验只报告下降是否出现、出现在哪两个测量点之间，不解释它的成因。加密测量点定位下降位置是动手练习第 3 题（选做）。

接下来看锁页内存带来多少增益。请对照 §9 的表格填出三个区间的对比：小规模段 v1 与 v2 是否基本重合？过渡段两者相差几倍？最大规模处相差几倍？

模型解释了前两段：锁页内存改善的是 $\beta$，因为它省掉的中转复制代价正比于数据量，要在 $S$ 足够大时才显现；判断多大算足够大的依据就是 $S_{1/2}$。小规模段两个版本重合，是因为此时时间由 $\alpha$ 支配，而两者的 $\alpha$ 接近——请在 §11 的输出里核对这一点。

最大规模处的差距要分两种情形，**两种在真实硬件上都会出现**：

- 若本机出现了骤降，锁页版本继续上升而可分页版本停在低位段上，两者相差可达一个数量级；
- 若本机没有骤降，两者趋近同一条渐近线，锁页的增益只有几个百分点。

因此工程结论必须写成有条件的形式：**锁页内存在小规模传输上几乎没有收益；大规模传输上是否有收益、收益多大，取决于该平台上可分页路径与总线各自的上限——两者相差越大，锁页的收益越明显。** 引用时必须用本机测出的倍数，不能沿用别处的数字，也不能把某一台机器上的倍数当成普遍规律。

**③ v3 与 v1 的曲线几乎重合，这是本实验的核心观察。**

v3 用的是 `aclrtMemcpyAsync`，接口名中带 Async，但它的曲线与用同步接口的 v1 几乎完全一致。请从 §11 的输出读出两者 H2D 方向的 $\beta$，比较它们相差百分之几；若本机出现了 ② 的下降，再对照两者下降的位置是否相同。§3.2 已经给出了原因：**当 Host 内存是非锁页内存时，`aclrtMemcpyAsync` 在内存复制任务完成后才返回。** v3 中的异步只是一个名称，接口内部退化成了同步。

这条观察在两台参考机上都成立，是本实验少数几条不随平台变化的结论之一。它要求学生修正一个常见的直觉：**异步不是由接口名称决定的，而是由内存类型决定的。** 四个版本中只有 v4 是真正异步的。

**④ 异步不降低单次传输的耗时，反而增大了固定开销。**

v4 与 v2 都用锁页内存，差别只在复制接口。请从本机 §11 的输出读出两组数：两者的 $\alpha$ 相差多少微秒？两者的 $\beta$ 是否接近？

$\alpha$ 变大在两台参考机上都成立，差值都在十微秒量级。这份额外开销来自异步路径本身：任务下发到 Stream、设备调度并执行、完成通知回传、`aclrtSynchronizeStream` 唤醒主机线程，比同步接口内部的等待多了若干环节——§3.2 的任务调度图把这条回路的每一个环节都编号画了出来。

$\beta$ 则要分两种情形，**两种在真实硬件上都会出现**：

- 在一部分平台上 v4 与 v2 的 $\beta$ 基本相同，异步的代价全部落在 $\alpha$ 上；
- 在另一部分平台上 v4 的 $\beta$ 明显低于 v2。此时代价不止体现在固定开销上，请一并记录，并回到 §9 的表格看 v4 的带宽在大规模段是否仍在单调上升——**若它在中途回落又回升，说明大规模段的点并没有落在渐近线上，解出的 $\beta$ 不能当作该版本的上限来引用。**

后果是 v4 在多数规模上慢于 v2。请从 §9 的表格判断两条曲线是否交叉、若交叉则在哪个规模上交叉，或者全程不交叉。本实验在下发之后紧接着就调用了 `aclrtSynchronizeStream`，让出的空隙没有被利用，因此只测到了异步的代价、没有测到它的收益。要把这段等待时间用起来，需要在等待期间下发别的任务，届时应当用**本机实测的 $\alpha$ 差值**判断重叠是否能够带来收益。

**⑤ 两个方向的带宽不对称，而固定开销对称。**

请从本机 §11 的输出读出锁页版本两个方向的 $\alpha$ 与 $\beta$：$\beta$ 上 H2D 与 D2H 相差几倍？$\alpha$ 是否基本相同？两台参考机上 H2D 的 $\beta$ 都高于 D2H、两个方向的 $\alpha$ 都基本相同——**方向上的定性关系是稳定的，但倍数差别很大**，因此这个倍数是本机的一项测量结果，不是一个可以引用的常数。

不对称只出现在与传输量成正比的那一项上，与固定开销无关。本实验只要求观察并记录这个差异，不解释它的成因——那需要更底层的性能计数器数据，本实验没有采集。

**⑥ $\alpha$ 的量级比第二章测到的 DRAM 访问延迟高约两个数量级。**

第二章测 DRAM 时 $\alpha$ 在纳秒量级，本实验的 $\alpha$ 在十微秒量级；具体取值见本机 §11 的输出，其中同步路径与异步路径之间可以相差数倍。这两个数量级的差距来自路径的长度：DRAM 访问只跨过内存控制器，而一次 H2D 传输要经过用户态接口调用、Runtime 的任务构造、驱动的系统调用、DMA 描述符的构造与下发、总线传输，以及完成通知的回传。同一个模型、同一套方法，在相差两个数量级的两种介质上都成立。

**⑦ $S_{1/2}$ 因版本而异，也因平台而异，引用时必须取本机本次的取值。**

请从本机 §11 的输出抄下八行的 $S_{1/2}$。差异的来源由定义直接可见：$S_{1/2} = \alpha\beta$，锁页提高 $\beta$、异步提高 $\alpha$，两者都使 $S_{1/2}$ 变大。**渐近带宽越高、固定开销越大，把固定开销分摊到可以忽略所需的传输量就越大。**

有两点需要留意。其一，**八行之间的相对关系随平台变化**：可分页版本的 $S_{1/2}$ 是否明显小于锁页版本，取决于 ② 中那条边界在本机是否存在——边界存在时可分页版本的 $\beta$ 被压在低位段上，$S_{1/2}$ 随之小得多；边界不存在时八行会落在同一个量级里。其二，**同一行的绝对值在两台参考机上相差数倍**。因此凡是要按 $S_{1/2}$ 决定传输块大小的场合，都必须取本机本次的取值，并且取与实际使用的内存类型、复制接口相对应的那一行。

---

### 🎓 结论

本实验建立了两条量化基础：**其一，主机与设备之间的传输代价由两个常数刻画——固定开销 $\alpha$ 与渐近带宽 $\beta$，由它们导出的半带宽规模 $S_{1/2} = \alpha\beta$ 是判断传输块应当划分多大的依据；其二，锁页内存与异步接口不是两个独立的选项，前者是后者的前提——`aclrtMemcpyAsync` 只有在锁页内存上才真正异步，而即便真正异步，单次传输也不会变快，反而要多付一份固定开销。异步的价值不在于让这一次传输更快，而在于让出等待时间供后续任务重叠使用。**

方法论上还有一条：两参数模型有它的适用范围。**先用 §11 的两条自检确认数据落在模型的适用范围内，再引用 $\alpha$ 与 $\beta$**，而不是默认模型成立。

最后一条与本节的写法有关。本节多条结论写成了分支判据，是因为它们在两台参考机上落到了不同的分支：同一段代码、同一个模型，在一台机器上可分页路径存在规模边界、锁页内存在大规模传输上的增益达到一个数量级；在另一台机器上四条曲线全程单调上升、锁页的增益只有几个百分点。**两种情形都是真实硬件上的正常结果，都不是测量错误。** 因此本实验交付给学生的不是某一组数字，而是一套判据：读哪一行、比哪两个数、比出来落在哪一侧、由此该采用哪一条结论。把别人机器上的数字直接引用过来，是最容易犯、也最难被发现的错误。


## 13. 🔧 动手练习

两道必做题合计约 35 分钟；第 3 题选做，只在本机出现带宽骤降时才有内容可做。

> **提示**：第 2、3 题要改 C++ 源码。修改源码后需要重新执行 `%%writefile` 单元格；由于 §7.1 为覆盖写、其后四个为追加写，**须从 7.1 开始按顺序重新执行**。

1. **标出半带宽规模**（约 10 分钟，只改 Python）

   由 §11 解出的 $\alpha$ 与 $\beta$ 算出各版本的 $S_{1/2}$，在 §10 的带宽曲线上用 `ax.axvline` 画出竖直参考线，并在 $y = \beta/2$ 处画一条水平线。按定义，两条线的交点应当落在该版本的实测曲线上。请指出四个版本中哪些落得准、哪些偏离，并说明偏离的那些与 §11 两条自检的结论是否对得上。

2. **二维复制与逐行复制的对比**（约 25 分钟）

   构造一个 $4096 \times 4096$ 的 `float` 矩阵（主机侧），把其中 $1024 \times 1024$ 的子块搬到设备。分别用两种方法实现并计时：

   - 方法一：循环 1024 次，每次 `aclrtMemcpy` 搬一行（4 KB）；
   - 方法二：一次 `aclrtMemcpy2d`，`spitch` 取 $4096 \times 4$ 字节、`dpitch` 取 $1024 \times 4$ 字节、`width` 取 $1024 \times 4$ 字节、`height` 取 1024。

   两种方法都用 `aclrtMallocHost` 申请主机内存、都用同步接口，因此 $\alpha$ 取 §11 中 v2 那一行的取值。先用它预测方法一比方法二多花多少时间——方法一多付了 1023 次固定开销——再与实测差额对照。若实测差额明显大于预测值，说明逐行调用还引入了固定开销之外的代价。

3. **（选做）定位可分页路径的带宽下降点**

   先看 §11 输出的「带宽骤降」一列：本机若为「无」，本题跳过。

   若为「有」，把 §7.5 的规模循环改为在骤降点前后各一个数量级的区间内以 256 KB 为步长取点，并只测 v1 与 v2 两个版本以缩短运行时间。回答两个问题：下降发生在哪一个规模上；v2 在同一区间是否也有下降。若 v2 没有，说明这个下降只发生在可分页这一条路径上。

   *实现提示*：规模循环要由按指数枚举改为按字节数枚举，`RepeatFor` 的分档条件相应调整。


## 14. 🤔 思考题

1. §3.2 指出，`aclrtMemcpyAsync` 在可分页内存上会退化为同步。请从 DMA 引擎的工作方式解释这个设计：为什么 DMA 要求内存在整个传输期间不被换出？

2. v3 与 v1 的带宽曲线几乎重合。假设一位同学没有核对接口的前提条件，只是把代码里的 `aclrtMemcpy` 全部换成 `aclrtMemcpyAsync` 并加上流同步，然后报告异步接口没有带来任何收益。这个结论错在哪里？正确的表述应该是什么？

3. 设想一个推理服务，每秒处理 100 张 224×224×3 的 `uint8` 图片。结合本实验解出的 $\alpha$、$\beta$ 与 $S_{1/2}$ 估算：单张图片的传输量是多少？它落在带宽曲线的哪一段？若把多张图片拼成一批再传，一批多少张才能让传输效率过半？


## 15. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 概念 | 要点 |
| --- | --- |
| 两侧内存 | Device 内存用 `aclrtMalloc` / `aclrtFree`，Host 内存用 `aclrtMallocHost` / `aclrtFreeHost`，**两对接口不可互换** |
| Device 内存的四条规则 | 首地址 64 字节对齐；size 不能为 0；内容不初始化；不宜频繁申请释放 |
| 可分页与锁页 | 可分页内存会被换出，传输需经中转缓冲区；**锁页内存映射固定，可直接经 DMA 传输，无需 CPU 参与** |
| 对齐要求 | `aclrtMemcpyAsync` 要求两端地址 64 字节对齐；acl 的两个分配器自动满足，`malloc` **不满足** |
| `aclrtMemcpy` 的五参数 | `destMax` 是目的缓冲区上限，`count` 是实际复制长度，二者语义不同；`kind` 当前为预留参数 |
| 异步的前提 | **`aclrtMemcpyAsync` 只有在锁页内存上才真正异步**；可分页内存上它在复制完成后才返回 |
| 异步的代价与价值 | 异步不降低单次传输耗时，还要多付一份固定开销；它让出的是等待时间，供后续任务重叠使用 |
| 传输时间模型 | $T = \alpha + S/\beta$；$\alpha$ 为单次固定开销，$\beta$ 为渐近带宽 |
| 模型的适用范围 | 该模型假定耗时随传输量线性增长。**在部分平台上，可分页路径跨过某个规模后带宽显著下降**，此时模型只能分段使用；引用 $\alpha$ 与 $\beta$ 之前，须先用 §11 的两条自检确认数据落在模型的适用范围内 |
| 半带宽规模 | $S_{1/2} = \alpha\beta$，有效带宽达到 $\beta/2$ 的传输量，是判断块该切多大的判据；四个版本取值不同、同一版本在不同平台上也相差数倍，引用时须取本机本次测出的那一个 |
| 结论的平台依赖 | 可分页路径是否存在规模边界、锁页内存在大规模传输上的增益有多大，两台参考机给出的答案相反；**这两种情形在真实硬件上都会出现**。引用 §12 的任何一条结论之前，须先按判据确定本机落在哪一侧 |
| 测量方法 | 首次触碰、预热、多次重复取平均、同步之后再停止计时、内存只申请一次 |
| 校验方式 | 内存复制用**逐字节比对**；相对误差阈值是浮点计算的工具，用在这里会掩盖错误 |
| 运行模式 | A2 与 A3 系列不支持 `ACL_DEVICE`，`aclrtGetRunMode` 恒返回 `ACL_HOST`（Runtime API 参考）；此时主机与设备内存独立，数据须显式复制 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">概念</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两侧内存</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 内存用 <code>aclrtMalloc</code> / <code>aclrtFree</code>，Host 内存用 <code>aclrtMallocHost</code> / <code>aclrtFreeHost</code>，<strong>两对接口不可互换</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Device 内存的四条规则</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首地址 64 字节对齐；size 不能为 0；内容不初始化；不宜频繁申请释放</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可分页与锁页</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可分页内存会被换出，传输需经中转缓冲区；<strong>锁页内存映射固定，可直接经 DMA 传输，无需 CPU 参与</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">对齐要求</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpyAsync</code> 要求两端地址 64 字节对齐；acl 的两个分配器自动满足，<code>malloc</code> <strong>不满足</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>aclrtMemcpy</code> 的五参数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>destMax</code> 是目的缓冲区上限，<code>count</code> 是实际复制长度，二者语义不同；<code>kind</code> 当前为预留参数</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步的前提</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong><code>aclrtMemcpyAsync</code> 只有在锁页内存上才真正异步</strong>；可分页内存上它在复制完成后才返回</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步的代价与价值</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">异步不降低单次传输耗时，还要多付一份固定开销；它让出的是等待时间，供后续任务重叠使用</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">传输时间模型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$T = \alpha + S/\beta$；$\alpha$ 为单次固定开销，$\beta$ 为渐近带宽</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型的适用范围</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">该模型假定耗时随传输量线性增长。<strong>在部分平台上，可分页路径跨过某个规模后带宽显著下降</strong>，此时模型只能分段使用；引用 $\alpha$ 与 $\beta$ 之前，须先用 §11 的两条自检确认数据落在模型的适用范围内</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">半带宽规模</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">$S_{1/2} = \alpha\beta$，有效带宽达到 $\beta/2$ 的传输量，是判断块该切多大的判据；四个版本取值不同、同一版本在不同平台上也相差数倍，引用时须取本机本次测出的那一个</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">结论的平台依赖</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可分页路径是否存在规模边界、锁页内存在大规模传输上的增益有多大，两台参考机给出的答案相反；<strong>这两种情形在真实硬件上都会出现</strong>。引用 §12 的任何一条结论之前，须先按判据确定本机落在哪一侧</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">测量方法</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">首次触碰、预热、多次重复取平均、同步之后再停止计时、内存只申请一次</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">校验方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">内存复制用<strong>逐字节比对</strong>；相对误差阈值是浮点计算的工具，用在这里会掩盖错误</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行模式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">A2 与 A3 系列不支持 <code>ACL_DEVICE</code>，<code>aclrtGetRunMode</code> 恒返回 <code>ACL_HOST</code>（Runtime API 参考）；此时主机与设备内存独立，数据须显式复制</td>
</tr>
</tbody>
</table>

### 一条贯穿本章的原则

> **异构应用的性能问题，首先是数据搬运问题，其次才是计算问题。**

第六章的同一条原则说的是「性能问题首先是数据摆放问题」，指的是 Global Memory 与片上缓冲之间的搬运。本章把这条原则搬到了系统尺度：主机与设备之间的搬运同样要花时间，同样有固定开销与带宽上限，同样需要用分块与重叠来掩盖。四个版本中没有任何一处改动涉及被搬运的数据本身，全部的性能差异都来自内存的类型与接口的选择。

### 本实验没有回答的问题

本实验的每一次测量都是下发一次、等它完成：主机线程在等待期间无所事事，设备在传输期间也没有计算任务。因此本实验只测到了异步接口的代价，没有测到它的收益——收益要在等待期间下发别的任务时才出现，本实验没有这样做。

另外两件事也留在本实验之外：一次传输被拆成多块交错下发时，总耗时如何随块数变化；以及多条 Stream 同时发起传输时，$\beta$ 是被分食还是各自独立。这两个问题都要以本实验解出的 $\alpha$、$\beta$ 与 $S_{1/2}$ 为输入，**而这三个数随平台变化，两台参考机上相差数倍，必须用本机当次运行的结果，不能沿用本节正文中的任何数字。**
